In [2]:
pip install pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

files = {
    "NBA": "Data/NBA Data.csv",
    "MLB": "Data/MLB Data.csv",
    "NFL": "Data/NFL Data.csv"
}

for league, filename in files.items():
    print("\n" + "=" * 80)
    print(f"{league} DATASET")
    print("=" * 80)

    df = pd.read_csv(filename)

    print(f"\nDimensions: {df.shape}")
    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nFirst 5 rows:")
    print(df.head().to_string(index=False))

    print("\nMissing values:")
    missing = df.isna().sum()
    print(missing[missing > 0].to_string()
          if missing.any() else "No missing values.")

    print(f"\nDuplicate rows: {df.duplicated().sum():,}")


NBA DATASET


/tmp/ipykernel_3601/867549914.py:14: DtypeWarning: Columns (15: gameSubLabel) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)



Dimensions: (71827, 17)

Column names:
['gameId', 'gameDate', 'hometeamCity', 'hometeamName', 'hometeamId', 'awayteamCity', 'awayteamName', 'awayteamId', 'homeScore', 'awayScore', 'winner', 'gameType', 'attendance', 'arenaId', 'gameLabel', 'gameSubLabel', 'seriesGameNumber']

Data types:
gameId                int64
gameDate                str
hometeamCity            str
hometeamName            str
hometeamId            int64
awayteamCity            str
awayteamName            str
awayteamId            int64
homeScore             int64
awayScore             int64
winner                int64
gameType                str
attendance          float64
arenaId               int64
gameLabel               str
gameSubLabel            str
seriesGameNumber    float64

First 5 rows:
  gameId            gameDate hometeamCity hometeamName  hometeamId awayteamCity awayteamName  awayteamId  homeScore  awayScore     winner gameType  attendance  arenaId        gameLabel gameSubLabel  seriesGameNumber
424

/tmp/ipykernel_3601/867549914.py:14: DtypeWarning: Columns (10: usedh, 11: htbf, 13: attendance, 17: temp, 20: oscorer, 21: forfeit, 27: umplf, 28: umprf) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)



Dimensions: (226221, 43)

Column names:
['gid', 'visteam', 'hometeam', 'site', 'date', 'number', 'starttime', 'daynight', 'innings', 'tiebreaker', 'usedh', 'htbf', 'timeofgame', 'attendance', 'fieldcond', 'precip', 'sky', 'temp', 'winddir', 'windspeed', 'oscorer', 'forfeit', 'suspend', 'umphome', 'ump1b', 'ump2b', 'ump3b', 'umplf', 'umprf', 'wp', 'lp', 'save', 'gametype', 'vruns', 'hruns', 'wteam', 'lteam', 'line', 'batteries', 'lineups', 'box', 'pbp', 'season']

Data types:
gid               str
visteam           str
hometeam          str
site              str
date            int64
number        float64
starttime         str
daynight          str
innings       float64
tiebreaker    float64
usedh          object
htbf           object
timeofgame    float64
attendance     object
fieldcond         str
precip            str
sky               str
temp           object
winddir           str
windspeed     float64
oscorer           str
forfeit           str
suspend       float64
umphome      

In [4]:
# CELL 1: Load the raw datasets and define reusable cleaning functions

import pandas as pd
import numpy as np
from pathlib import Path

# Load the three original CSV files.
# low_memory=False prevents pandas from guessing different data types
# in different chunks of a large file.
nba_raw = pd.read_csv("Data/NBA Data.csv", low_memory=False)
mlb_raw = pd.read_csv("Data/MLB Data.csv", low_memory=False)
nfl_raw = pd.read_csv("Data/NFL Data.csv", low_memory=False)

# Keep the raw DataFrames unchanged. We will create separate cleaned copies.
nba = nba_raw.copy()
mlb = mlb_raw.copy()
nfl = nfl_raw.copy()

print("RAW DATASET DIMENSIONS")
print(f"NBA: {nba.shape[0]:,} rows, {nba.shape[1]} columns")
print(f"MLB: {mlb.shape[0]:,} rows, {mlb.shape[1]} columns")
print(f"NFL: {nfl.shape[0]:,} rows, {nfl.shape[1]} columns")


def attendance_audit(df, attendance_col, league):
    """
    Summarize attendance quality before cleaning.

    We distinguish:
    - Missing values already recorded as missing
    - Values that cannot be converted to numbers
    - Zero attendance
    - Negative attendance, which is invalid for our purposes
    - Valid positive attendance
    """
    original = df[attendance_col]
    numeric = pd.to_numeric(original, errors="coerce")

    print(f"\n--- {league}: ATTENDANCE AUDIT ---")
    print(f"Total rows: {len(df):,}")
    print(f"Originally missing: {original.isna().sum():,}")
    print(f"Missing/non-numeric after conversion: {numeric.isna().sum():,}")
    print(
        "Non-numeric values among originally nonmissing entries:",
        int((original.notna() & numeric.isna()).sum())
    )
    print(f"Zero attendance: {(numeric == 0).sum():,}")
    print(f"Negative attendance: {(numeric < 0).sum():,}")
    print(f"Positive numeric attendance: {(numeric > 0).sum():,}")
    print(f"Minimum numeric attendance: {numeric.min()}")
    print(f"Maximum numeric attendance: {numeric.max()}")

    # Show examples of entries that could not be converted to numbers.
    invalid_entries = original[original.notna() & numeric.isna()]
    if not invalid_entries.empty:
        print("Examples of nonnumeric attendance values:")
        print(invalid_entries.astype(str).value_counts().head(10))


def clean_attendance(df, attendance_col, league):
    """
    Convert attendance to numeric and remove rows with:
    - Missing or nonnumeric attendance
    - Zero attendance
    - Negative attendance

    We retain only strictly positive numeric attendance.
    The function returns a new DataFrame and reports row counts.
    """
    cleaned = df.copy()
    rows_before = len(cleaned)

    # Invalid text entries become NaN, so they cannot be mistaken for attendance.
    cleaned[attendance_col] = pd.to_numeric(
        cleaned[attendance_col], errors="coerce"
    )

    # Retain only rows with positive attendance.
    cleaned = cleaned.loc[cleaned[attendance_col] > 0].copy()

    rows_after = len(cleaned)
    print(f"\n--- {league}: ATTENDANCE CLEANING ---")
    print(f"Rows before: {rows_before:,}")
    print(f"Rows retained: {rows_after:,}")
    print(f"Rows removed: {rows_before - rows_after:,}")
    print(f"Percent retained: {100 * rows_after / rows_before:.2f}%")

    return cleaned


print("\nHelper functions defined. No rows have been removed yet.")

RAW DATASET DIMENSIONS
NBA: 71,827 rows, 17 columns
MLB: 226,221 rows, 43 columns
NFL: 4,808 rows, 66 columns

Helper functions defined. No rows have been removed yet.


In [5]:
# CELL 2: Attendance audit, date standardization, and initial cleaning

# ------------------------------------------------------------
# PART A: Audit attendance BEFORE removing any observations
# ------------------------------------------------------------

attendance_audit(nba, "attendance", "NBA")
attendance_audit(mlb, "attendance", "MLB")
attendance_audit(nfl, "weekly_attendance_home", "NFL")


# ------------------------------------------------------------
# PART B: Standardize dates
# ------------------------------------------------------------

# NBA: Convert gameDate to datetime.
# errors="coerce" converts unparseable dates to NaT, which we will count.
nba["gameDate"] = pd.to_datetime(nba["gameDate"], errors="coerce")

print("\n--- NBA DATE CHECK ---")
print("Unparseable/missing dates:", nba["gameDate"].isna().sum())
print("Date range:", nba["gameDate"].min(), "to", nba["gameDate"].max())


# MLB: The date field is stored as an integer in YYYYMMDD format.
# Convert it to a string first, then parse it as a date.
# Int64 allows missing values without forcing the entire column to float.
mlb["date"] = pd.to_datetime(
    mlb["date"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

print("\n--- MLB DATE CHECK ---")
print("Unparseable/missing dates:", mlb["date"].isna().sum())
print("Date range before the 1960 cutoff:",
      mlb["date"].min(), "to", mlb["date"].max())


# NFL: The dataset has a separate year and date string.
# Example date string: "Sun September 23".
# We remove the weekday prefix, then append the year.
# This assumes the year column identifies the calendar year of the game.
nfl["date_home_original"] = nfl["date_home"].astype("string")

# Remove a leading weekday name, such as "Sun ", "Mon ", or "Thu ".
nfl["date_home_month_day"] = nfl["date_home_original"].str.replace(
    r"^[A-Za-z]{3,9}\s+", "", regex=True
)

# Combine the month/day text with the numeric year.
nfl["gameDate"] = pd.to_datetime(
    nfl["date_home_month_day"].astype("string")
    + " "
    + nfl["year"].astype("Int64").astype("string"),
    errors="coerce"
)

print("\n--- NFL DATE CHECK ---")
print("Unparseable/missing reconstructed dates:", nfl["gameDate"].isna().sum())
print("Date range:", nfl["gameDate"].min(), "to", nfl["gameDate"].max())

# Print examples of NFL dates that did not parse, if any.
nfl_bad_dates = nfl.loc[
    nfl["gameDate"].isna(),
    ["year", "date_home", "date_home_month_day"]
]
if not nfl_bad_dates.empty:
    print("\nExamples of NFL dates that did not parse:")
    print(nfl_bad_dates.head(15).to_string(index=False))


# ------------------------------------------------------------
# PART C: Convert key numeric columns
# ------------------------------------------------------------

# These conversions make it easier to calculate score differences,
# define close games, and run regressions later.
#
# errors="coerce" is intentional: unexpected text becomes missing
# rather than being silently treated as a number.

# NBA numeric columns that may be used in the analysis.
nba_numeric_cols = [
    "homeScore", "awayScore", "attendance",
    "hometeamId", "awayteamId"
]
for col in nba_numeric_cols:
    if col in nba.columns:
        nba[col] = pd.to_numeric(nba[col], errors="coerce")

# MLB scores and attendance.
mlb_numeric_cols = ["vruns", "hruns", "attendance", "season"]
for col in mlb_numeric_cols:
    if col in mlb.columns:
        mlb[col] = pd.to_numeric(mlb[col], errors="coerce")

# NFL game-level attendance and scores.
# The meaning of some other NFL columns needs further investigation
# before we decide whether they belong in the model.
nfl_numeric_cols = [
    "weekly_attendance_home",
    "year", "week",
    "total_home", "total_away"
]
for col in nfl_numeric_cols:
    if col in nfl.columns:
        nfl[col] = pd.to_numeric(nfl[col], errors="coerce")


# ------------------------------------------------------------
# PART D: Apply attendance cleaning
# ------------------------------------------------------------

nba_clean = clean_attendance(nba, "attendance", "NBA")
mlb_clean = clean_attendance(mlb, "attendance", "MLB")
nfl_clean = clean_attendance(nfl, "weekly_attendance_home", "NFL")


# ------------------------------------------------------------
# PART E: Apply MLB historical cutoff
# ------------------------------------------------------------

# Use January 1, 1960 as the initial cutoff.
# This is a proposed research-design choice, not a universal rule.
# We will inspect the retained sample before finalizing it.
mlb_rows_before_cutoff = len(mlb_clean)

mlb_clean = mlb_clean.loc[
    mlb_clean["date"].notna()
    & (mlb_clean["date"] >= pd.Timestamp("1960-01-01"))
].copy()

print("\n--- MLB HISTORICAL CUTOFF ---")
print("Rows after attendance cleaning, before date cutoff:",
      f"{mlb_rows_before_cutoff:,}")
print("Rows retained from 1960 onward:", f"{len(mlb_clean):,}")
print("Rows removed by date cutoff:",
      f"{mlb_rows_before_cutoff - len(mlb_clean):,}")
print("Final MLB date range:",
      mlb_clean["date"].min(), "to", mlb_clean["date"].max())


# ------------------------------------------------------------
# PART F: Report final cleaning totals
# ------------------------------------------------------------

print("\n--- CLEANING SUMMARY ---")
print(f"NBA: {len(nba_raw):,} raw rows -> {len(nba_clean):,} rows")
print(f"MLB: {len(mlb_raw):,} raw rows -> {len(mlb_clean):,} rows")
print(f"NFL: {len(nfl_raw):,} raw rows -> {len(nfl_clean):,} rows")

print("\nAttendance is now numeric in all three cleaned datasets.")
print("NBA and NFL include all game types present in the source files.")
print("MLB is restricted to games dated January 1, 1960 or later.")
print("Original raw DataFrames remain available.")


--- NBA: ATTENDANCE AUDIT ---
Total rows: 71,827
Originally missing: 5,029
Missing/non-numeric after conversion: 5,029
Non-numeric values among originally nonmissing entries: 0
Zero attendance: 0
Negative attendance: 0
Positive numeric attendance: 66,798
Minimum numeric attendance: 9.0
Maximum numeric attendance: 200049.0

--- MLB: ATTENDANCE AUDIT ---
Total rows: 226,221
Originally missing: 3,158
Missing/non-numeric after conversion: 3,160
Non-numeric values among originally nonmissing entries: 2
Zero attendance: 20,955
Negative attendance: 0
Positive numeric attendance: 202,106
Minimum numeric attendance: 0.0
Maximum numeric attendance: 92706.0
Examples of nonnumeric attendance values:
attendance
6500?    1
<1000    1
Name: count, dtype: int64

--- NFL: ATTENDANCE AUDIT ---
Total rows: 4,808
Originally missing: 0
Missing/non-numeric after conversion: 0
Non-numeric values among originally nonmissing entries: 0
Zero attendance: 0
Negative attendance: 0
Positive numeric attendance: 4,8

/tmp/ipykernel_3601/1362862334.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nfl["gameDate"] = pd.to_datetime(



--- NBA: ATTENDANCE CLEANING ---
Rows before: 71,827
Rows retained: 66,798
Rows removed: 5,029
Percent retained: 93.00%

--- MLB: ATTENDANCE CLEANING ---
Rows before: 226,221
Rows retained: 202,106
Rows removed: 24,115
Percent retained: 89.34%

--- NFL: ATTENDANCE CLEANING ---
Rows before: 4,808
Rows retained: 4,808
Rows removed: 0
Percent retained: 100.00%

--- MLB HISTORICAL CUTOFF ---
Rows after attendance cleaning, before date cutoff: 202,106
Rows retained from 1960 onward: 135,624
Rows removed by date cutoff: 66,482
Final MLB date range: 1960-04-12 00:00:00 to 2025-11-01 00:00:00

--- CLEANING SUMMARY ---
NBA: 71,827 raw rows -> 66,798 rows
MLB: 226,221 raw rows -> 135,624 rows
NFL: 4,808 raw rows -> 4,808 rows

Attendance is now numeric in all three cleaned datasets.
NBA and NFL include all game types present in the source files.
MLB is restricted to games dated January 1, 1960 or later.
Original raw DataFrames remain available.


In [6]:
# CELL 3: Check row structure, identifiers, dates, and score consistency

# These checks help us understand what one row represents in each dataset.
# They are diagnostic only: this cell does not delete any additional rows.


# ------------------------------------------------------------
# PART A: NBA structure
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NBA: ROW STRUCTURE")
print("=" * 65)

print("Rows:", f"{len(nba_clean):,}")
print("Unique game IDs:", f"{nba_clean['gameId'].nunique():,}")
print("Rows with a game ID already seen:",
      f"{nba_clean['gameId'].duplicated().sum():,}")
print("Exact duplicate rows:", f"{nba_clean.duplicated().sum():,}")

# A game should generally have two identifiable teams and a score for each.
print("Missing home team IDs:", nba_clean["hometeamId"].isna().sum())
print("Missing away team IDs:", nba_clean["awayteamId"].isna().sum())
print("Missing home scores:", nba_clean["homeScore"].isna().sum())
print("Missing away scores:", nba_clean["awayScore"].isna().sum())

# Examine the game types without excluding any.
print("\nNBA gameType values:")
print(nba_clean["gameType"].value_counts(dropna=False).head(20))

# Check whether repeated game IDs correspond to multiple rows.
nba_duplicate_ids = nba_clean.loc[
    nba_clean["gameId"].duplicated(keep=False)
].sort_values(["gameId", "gameDate"])

if not nba_duplicate_ids.empty:
    print("\nExamples of repeated NBA game IDs:")
    print(
        nba_duplicate_ids[
            ["gameId", "gameDate", "hometeamName", "awayteamName",
             "homeScore", "awayScore", "attendance", "gameType"]
        ].head(20).to_string(index=False)
    )


# ------------------------------------------------------------
# PART B: MLB structure
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MLB: ROW STRUCTURE")
print("=" * 65)

print("Rows:", f"{len(mlb_clean):,}")
print("Unique game IDs:", f"{mlb_clean['gid'].nunique():,}")
print("Rows with a game ID already seen:",
      f"{mlb_clean['gid'].duplicated().sum():,}")
print("Exact duplicate rows:", f"{mlb_clean.duplicated().sum():,}")

print("Missing dates:", mlb_clean["date"].isna().sum())
print("Missing visiting teams:", mlb_clean["visteam"].isna().sum())
print("Missing home teams:", mlb_clean["hometeam"].isna().sum())
print("Missing visitor scores:", mlb_clean["vruns"].isna().sum())
print("Missing home scores:", mlb_clean["hruns"].isna().sum())

# Game type values may distinguish regular-season and postseason games.
# We report them but do not exclude any type.
print("\nMLB gametype values:")
print(mlb_clean["gametype"].value_counts(dropna=False))

# Check whether the winning and losing team fields agree with the scores.
# These comparisons are diagnostic; they do not delete observations.
mlb_has_scores = mlb_clean["vruns"].notna() & mlb_clean["hruns"].notna()

print(
    "Games with tied recorded scores:",
    (mlb_has_scores & (mlb_clean["vruns"] == mlb_clean["hruns"])).sum()
)

mlb_duplicate_ids = mlb_clean.loc[
    mlb_clean["gid"].duplicated(keep=False)
].sort_values(["gid", "date"])

if not mlb_duplicate_ids.empty:
    print("\nExamples of repeated MLB game IDs:")
    print(
        mlb_duplicate_ids[
            ["gid", "date", "visteam", "hometeam",
             "vruns", "hruns", "attendance", "gametype"]
        ].head(20).to_string(index=False)
    )


# ------------------------------------------------------------
# PART C: NFL structure
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NFL: ROW STRUCTURE")
print("=" * 65)

print("Rows:", f"{len(nfl_clean):,}")
print("Exact duplicate rows:", f"{nfl_clean.duplicated().sum():,}")
print("Missing reconstructed dates:", nfl_clean["gameDate"].isna().sum())

# Report date coverage and games by season.
print("Date range:", nfl_clean["gameDate"].min(), "to",
      nfl_clean["gameDate"].max())
print("\nRows per year (first and last 10 years shown):")
rows_by_year = nfl_clean.groupby("year").size().sort_index()
print(rows_by_year.head(10))
print(rows_by_year.tail(10))

# Check whether the apparent home and away team fields are consistent.
# This is especially important because the dataset contains both
# team_name_home/team_name_away and home/away city/name fields.
team_columns = [
    "team_name_home", "team_name_away",
    "home_team_name_home", "away_team_name_home",
    "home_team_name_away", "away_team_name_away"
]
available_team_columns = [
    col for col in team_columns if col in nfl_clean.columns
]

print("\nExamples of NFL team fields:")
print(nfl_clean[available_team_columns].head(12).to_string(index=False))

# Check repeated year-week-team combinations.
# This is a diagnostic, not proof that a row is duplicated:
# teams can have unusual schedules, and the dataset's row structure
# still needs to be established.
if {"team_name_home", "year", "week"}.issubset(nfl_clean.columns):
    nfl_team_week_counts = (
        nfl_clean.groupby(["team_name_home", "year", "week"], dropna=False)
        .size()
    )
    print(
        "\nRepeated team_name_home/year/week combinations:",
        int((nfl_team_week_counts > 1).sum())
    )
    print("Largest number of rows in any such combination:",
          int(nfl_team_week_counts.max()))

# Show the game-level score fields, if present.
score_columns = [
    col for col in [
        "team_name_home", "team_name_away", "year", "week", "gameDate",
        "weekly_attendance_home", "total_home", "total_away",
        "home_home", "away_home", "home_away", "away_away"
    ]
    if col in nfl_clean.columns
]

print("\nExample NFL rows:")
print(nfl_clean[score_columns].head(12).to_string(index=False))


# ------------------------------------------------------------
# PART D: Check dates and attendance after cleaning
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL BASIC VALIDATION")
print("=" * 65)

for league, df, date_col, attendance_col in [
    ("NBA", nba_clean, "gameDate", "attendance"),
    ("MLB", mlb_clean, "date", "attendance"),
    ("NFL", nfl_clean, "gameDate", "weekly_attendance_home"),
]:
    print(f"\n{league}")
    print("Rows:", f"{len(df):,}")
    print("Missing dates:", f"{df[date_col].isna().sum():,}")
    print("Missing attendance:", f"{df[attendance_col].isna().sum():,}")
    print("Zero attendance:", f"{(df[attendance_col] == 0).sum():,}")
    print("Negative attendance:", f"{(df[attendance_col] < 0).sum():,}")
    print("Minimum attendance:", df[attendance_col].min())
    print("Maximum attendance:", df[attendance_col].max())

print("\nDiagnostic checks complete. No additional rows were deleted.")


NBA: ROW STRUCTURE
Rows: 66,798
Unique game IDs: 66,798
Rows with a game ID already seen: 0
Exact duplicate rows: 0
Missing home team IDs: 0
Missing away team IDs: 0
Missing home scores: 0
Missing away scores: 0

NBA gameType values:
gameType
Regular Season        61148
Playoffs               3973
Preseason              1582
NBA Emirates Cup         66
Play-in Tournament       28
NBA Cup                   1
Name: count, dtype: int64

MLB: ROW STRUCTURE
Rows: 135,624
Unique game IDs: 135,624
Rows with a game ID already seen: 0
Exact duplicate rows: 0
Missing dates: 0
Missing visiting teams: 0
Missing home teams: 0
Missing visitor scores: 0
Missing home scores: 0

MLB gametype values:
gametype
regular           134031
lcs                  578
divisionseries       498
worldseries          378
allstar               70
wildcard              55
playoff               14
Name: count, dtype: int64
Games with tied recorded scores: 89

NFL: ROW STRUCTURE
Rows: 4,808
Exact duplicate rows: 0
Missi

In [7]:
# CELL 4: Fix NFL dates and investigate unusual observations
#
# This cell creates updated cleaned DataFrames in memory.
# It does not modify or overwrite any CSV files.


# ------------------------------------------------------------
# PART A: Correct NFL date reconstruction
# ------------------------------------------------------------

# Start from the original NFL date text, not the incorrectly
# reconstructed gameDate column.
nfl_clean["date_home_original"] = nfl_clean["date_home"].astype("string")

# Remove a weekday prefix ONLY if the first token is a known weekday.
# The previous regex was too broad: it incorrectly removed month names
# such as "September" and "October".
nfl_clean["date_home_month_day"] = (
    nfl_clean["date_home_original"]
    .str.replace(
        r"^(Mon|Tue|Wed|Thu|Fri|Sat|Sun)(day)?\s+",
        "",
        regex=True,
        case=False
    )
    .str.strip()
)

# Build a date string such as "September 23 2001".
# Parse using an explicit format to avoid ambiguous date interpretation.
nfl_clean["gameDate"] = pd.to_datetime(
    nfl_clean["date_home_month_day"]
    + " "
    + nfl_clean["year"].astype("Int64").astype("string"),
    format="%B %d %Y",
    errors="coerce"
)

print("--- NFL DATE RECONSTRUCTION ---")
print("Rows:", len(nfl_clean))
print("Missing/unparseable reconstructed dates:",
      nfl_clean["gameDate"].isna().sum())
print("Date range:",
      nfl_clean["gameDate"].min(), "to", nfl_clean["gameDate"].max())

print("\nExamples of original and reconstructed NFL dates:")
print(
    nfl_clean[
        ["year", "date_home", "date_home_month_day", "gameDate"]
    ].head(15).to_string(index=False)
)

# Show any remaining failures so we can investigate them individually.
nfl_bad_dates = nfl_clean.loc[
    nfl_clean["gameDate"].isna(),
    ["year", "date_home", "date_home_month_day", "week"]
]
if not nfl_bad_dates.empty:
    print("\nRemaining NFL date parsing failures:")
    print(nfl_bad_dates.head(25).to_string(index=False))


# ------------------------------------------------------------
# PART B: Inspect extreme NBA attendance observations
# ------------------------------------------------------------

# Show the 15 lowest and 15 highest recorded attendance values.
# This helps distinguish plausible small crowds from possible data errors.
print("\n" + "=" * 65)
print("NBA: LOWEST ATTENDANCE OBSERVATIONS")
print("=" * 65)

nba_extreme_cols = [
    "gameDate", "hometeamCity", "hometeamName",
    "awayteamCity", "awayteamName",
    "homeScore", "awayScore", "attendance", "gameType"
]
nba_extreme_cols = [c for c in nba_extreme_cols if c in nba_clean.columns]

print(
    nba_clean.nsmallest(15, "attendance")[nba_extreme_cols]
    .to_string(index=False)
)

print("\nNBA: HIGHEST ATTENDANCE OBSERVATIONS")
print(
    nba_clean.nlargest(15, "attendance")[nba_extreme_cols]
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART C: Inspect extreme MLB attendance observations
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MLB: LOWEST ATTENDANCE OBSERVATIONS")
print("=" * 65)

mlb_extreme_cols = [
    "date", "visteam", "hometeam",
    "vruns", "hruns", "attendance", "gametype", "season"
]
mlb_extreme_cols = [c for c in mlb_extreme_cols if c in mlb_clean.columns]

print(
    mlb_clean.nsmallest(15, "attendance")[mlb_extreme_cols]
    .to_string(index=False)
)

print("\nMLB: HIGHEST ATTENDANCE OBSERVATIONS")
print(
    mlb_clean.nlargest(15, "attendance")[mlb_extreme_cols]
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART D: Inspect the NFL data's possible game-level fields
# ------------------------------------------------------------

# We need to distinguish game-level information from season-level totals.
# Do not calculate a game's margin using total_home or total_away:
# the sample suggests these are season aggregates.
#
# Print the full column list so we can identify candidate fields.
print("\n" + "=" * 65)
print("NFL: FULL COLUMN LIST")
print("=" * 65)

for i, col in enumerate(nfl_clean.columns, start=1):
    print(f"{i:02d}. {col}")


# Print representative rows with columns likely to relate to results.
# These are exploratory only; we are not yet treating any of them
# as the actual game score or game winner.
nfl_candidate_cols = [
    "gameDate", "year", "week",
    "team_name_home", "team_name_away",
    "weekly_attendance_home",
    "winner_home", "winner_away",
    "tie_home", "tie_away",
    "pts_win_home", "pts_loss_home",
    "pts_win_away", "pts_loss_away",
    "points_for_home", "points_against_home",
    "points_for_away", "points_against_away",
    "margin_of_victory_home", "margin_of_victory_away",
    "playoffs_home", "playoffs_away"
]
nfl_candidate_cols = [
    c for c in nfl_candidate_cols if c in nfl_clean.columns
]

print("\nNFL candidate result fields:")
print(nfl_clean[nfl_candidate_cols].head(20).to_string(index=False))


# Examine the unique values of a few candidate result fields.
# This helps determine whether they describe individual games or seasons.
for col in [
    "winner_home", "winner_away", "tie_home", "tie_away",
    "playoffs_home", "playoffs_away"
]:
    if col in nfl_clean.columns:
        print(f"\nNFL values for {col}:")
        print(nfl_clean[col].value_counts(dropna=False).head(15))


# Check whether purported season-level statistics are repeated for each
# team across that team's home-game rows.
# If these fields are constant within team-season, they are season-level,
# not game-level, and should not be used to infer individual game margins.
for col in [
    "total_home", "total_away",
    "points_for_home", "points_against_home",
    "margin_of_victory_home"
]:
    if col in nfl_clean.columns:
        counts = (
            nfl_clean.groupby(["team_name_home", "year"])[col]
            .nunique(dropna=False)
        )
        print(
            f"\n{col}: team-seasons with more than one value = "
            f"{int((counts > 1).sum()):,} of {len(counts):,}"
        )

print("\nCell 4 complete. No additional observations were removed.")

--- NFL DATE RECONSTRUCTION ---
Rows: 4808
Missing/unparseable reconstructed dates: 0
Date range: 2001-01-06 00:00:00 to 2019-12-29 00:00:00

Examples of original and reconstructed NFL dates:
 year    date_home date_home_month_day   gameDate
 2001 September 23        September 23 2001-09-23
 2001 September 30        September 30 2001-09-30
 2001   October 21          October 21 2001-10-21
 2001   November 4          November 4 2001-11-04
 2001  November 11         November 11 2001-11-11
 2001  November 18         November 18 2001-11-18
 2001   December 9          December 9 2001-12-09
 2001  December 23         December 23 2001-12-23
 2001 September 23        September 23 2001-09-23
 2001    October 7           October 7 2001-10-07
 2001   October 14          October 14 2001-10-14
 2001   November 4          November 4 2001-11-04
 2001  November 11         November 11 2001-11-11
 2001   December 2          December 2 2001-12-02
 2001   December 9          December 9 2001-12-09

NBA: LO

In [8]:
# CELL 5: Validate game results and calculate preliminary margins
#
# Purpose:
# 1. Verify the NFL's game-level score fields.
# 2. Calculate absolute margin of victory for each game.
# 3. Inspect the margin distributions before defining "close."
#
# No additional observations are removed in this cell.


# ------------------------------------------------------------
# PART A: NBA game margins
# ------------------------------------------------------------

# The NBA data directly record home and away scores.
# Absolute margin is the absolute difference between those scores.
nba_clean["score_margin"] = (
    nba_clean["homeScore"] - nba_clean["awayScore"]
)
nba_clean["abs_margin"] = nba_clean["score_margin"].abs()

print("=" * 65)
print("NBA: GAME MARGIN CHECK")
print("=" * 65)

print("Missing absolute margins:", nba_clean["abs_margin"].isna().sum())
print("Games with tied scores:", (nba_clean["abs_margin"] == 0).sum())
print("\nAbsolute margin summary:")
print(nba_clean["abs_margin"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]))

print("\nSmallest NBA margins:")
print(
    nba_clean.nsmallest(
        10, "abs_margin"
    )[["gameDate", "hometeamName", "awayteamName",
       "homeScore", "awayScore", "abs_margin", "gameType"]]
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART B: MLB game margins
# ------------------------------------------------------------

# The MLB data directly record visitor and home runs.
# Absolute margin is the absolute difference between those scores.
mlb_clean["score_margin"] = mlb_clean["hruns"] - mlb_clean["vruns"]
mlb_clean["abs_margin"] = mlb_clean["score_margin"].abs()

print("\n" + "=" * 65)
print("MLB: GAME MARGIN CHECK")
print("=" * 65)

print("Missing absolute margins:", mlb_clean["abs_margin"].isna().sum())
print("Games with tied recorded scores:", (mlb_clean["abs_margin"] == 0).sum())
print("\nAbsolute margin summary:")
print(mlb_clean["abs_margin"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]))

print("\nSmallest MLB margins:")
print(
    mlb_clean.nsmallest(
        10, "abs_margin"
    )[["date", "visteam", "hometeam",
       "vruns", "hruns", "abs_margin", "gametype"]]
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART C: NFL score-field validation
# ------------------------------------------------------------

# The NFL file appears to store the winning team's score in pts_win_home
# and the losing team's score in pts_loss_home. The "_home" suffix does
# not necessarily mean that these are the home team's points.
#
# We test whether:
# - the two score fields are present and sensible;
# - winner_home matches one of the two teams;
# - winner_home and winner_away agree;
# - the winning score is at least the losing score.
#
# We will not use season-level totals to calculate individual game margins.

print("\n" + "=" * 65)
print("NFL: GAME SCORE VALIDATION")
print("=" * 65)

# Convert these candidate game score fields to numeric, if not already.
for col in ["pts_win_home", "pts_loss_home", "pts_win_away", "pts_loss_away"]:
    if col in nfl_clean.columns:
        nfl_clean[col] = pd.to_numeric(nfl_clean[col], errors="coerce")

# Check whether both sets of score fields match.
# If they do, the "_home" and "_away" versions are redundant copies.
score_pair_cols = [
    "pts_win_home", "pts_loss_home",
    "pts_win_away", "pts_loss_away"
]
if all(col in nfl_clean.columns for col in score_pair_cols):
    print(
        "Rows where winner scores differ across home/away versions:",
        (
            (nfl_clean["pts_win_home"] != nfl_clean["pts_win_away"])
            | (nfl_clean["pts_loss_home"] != nfl_clean["pts_loss_away"])
        ).sum()
    )

# Check whether the winner labels agree and identify the winning team.
print(
    "Rows where winner_home differs from winner_away:",
    (nfl_clean["winner_home"] != nfl_clean["winner_away"]).sum()
)

winner_matches_home = (
    nfl_clean["winner_home"] == nfl_clean["team_name_home"]
)
winner_matches_away = (
    nfl_clean["winner_home"] == nfl_clean["team_name_away"]
)

print("Rows where listed winner matches home team:", winner_matches_home.sum())
print("Rows where listed winner matches away team:", winner_matches_away.sum())
print(
    "Rows where listed winner matches neither team:",
    (~winner_matches_home & ~winner_matches_away).sum()
)

# Check whether the purported winning score is at least the losing score.
valid_score_rows = (
    nfl_clean["pts_win_home"].notna()
    & nfl_clean["pts_loss_home"].notna()
)
print(
    "Rows with winning score lower than losing score:",
    (
        valid_score_rows
        & (nfl_clean["pts_win_home"] < nfl_clean["pts_loss_home"])
    ).sum()
)

print("\nExample NFL game results:")
print(
    nfl_clean[
        ["gameDate", "week", "team_name_home", "team_name_away",
         "winner_home", "pts_win_home", "pts_loss_home",
         "winner_away", "pts_win_away", "pts_loss_away"]
    ].head(20).to_string(index=False)
)

# Construct a preliminary absolute margin using the two score fields.
# This is appropriate only if the validation checks above support
# interpreting these fields as the winning and losing team's scores.
nfl_clean["abs_margin"] = (
    nfl_clean["pts_win_home"] - nfl_clean["pts_loss_home"]
).abs()

print("\nNFL absolute margin summary:")
print(nfl_clean["abs_margin"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]))

print("\nSmallest NFL margins:")
print(
    nfl_clean.nsmallest(
        15, "abs_margin"
    )[["gameDate", "week", "team_name_home", "team_name_away",
       "winner_home", "pts_win_home", "pts_loss_home", "abs_margin"]]
    .to_string(index=False)
)


# ------------------------------------------------------------
# PART D: Compare candidate close-game thresholds
# ------------------------------------------------------------

# Show the 10th, 15th, and 20th percentiles of absolute margin.
# These are descriptive candidates only. We will select the threshold
# after seeing the distributions and confirming the NFL fields.
#
# The 10th percentile means that approximately 10% of games have
# an absolute margin at or below that value.

for league, df in [
    ("NBA", nba_clean),
    ("MLB", mlb_clean),
    ("NFL", nfl_clean)
]:
    valid_margins = df["abs_margin"].dropna()

    print(f"\n{league}: ABSOLUTE MARGIN CUTOFFS")
    print("Valid games:", f"{len(valid_margins):,}")
    print("10th percentile:", valid_margins.quantile(0.10))
    print("15th percentile:", valid_margins.quantile(0.15))
    print("20th percentile:", valid_margins.quantile(0.20))
    print("25th percentile:", valid_margins.quantile(0.25))

print("\nCell 5 complete. No close-game indicator has been created yet.")

NBA: GAME MARGIN CHECK
Missing absolute margins: 0
Games with tied scores: 2

Absolute margin summary:
count    66798.000000
mean        11.011587
std          8.151054
min          0.000000
10%          2.000000
25%          5.000000
50%          9.000000
75%         15.000000
90%         22.000000
max         73.000000
Name: abs_margin, dtype: float64

Smallest NBA margins:
           gameDate  hometeamName awayteamName  homeScore  awayScore  abs_margin       gameType
2010-10-29 20:30:00     Mavericks    Grizzlies         90         90           0 Regular Season
2004-11-13 22:00:00 Trail Blazers      Raptors        105        105           0 Regular Season
2025-04-27 13:00:00       Pistons       Knicks         93         94           1       Playoffs
2025-04-13 13:00:00          Heat      Wizards        118        119           1 Regular Season
2025-04-11 22:00:00         Kings     Clippers        100        101           1 Regular Season
2025-04-04 20:00:00         Spurs    Cavalier

In [9]:
# CELL 6: Exclude preseason games and recheck game-level data
#
# Goals:
# 1. Remove NBA preseason games.
# 2. Check MLB game-type labels for preseason/spring-training games.
# 3. Check whether the NFL dataset appears to contain postseason weeks.
# 4. Investigate tied-score observations.
#
# This cell updates the cleaned DataFrames in memory only.


# ------------------------------------------------------------
# PART A: NBA — remove preseason games
# ------------------------------------------------------------

nba_before = len(nba_clean)

# Remove only explicitly labeled preseason games.
# Keep regular-season games, playoffs, play-in games, and cup games.
nba_clean = nba_clean.loc[
    nba_clean["gameType"].astype("string").str.strip().str.lower()
    != "preseason"
].copy()

print("=" * 65)
print("NBA: PRESEASON FILTER")
print("=" * 65)
print("Rows before:", f"{nba_before:,}")
print("Rows after:", f"{len(nba_clean):,}")
print("Rows removed:", f"{nba_before - len(nba_clean):,}")
print("\nRemaining NBA game types:")
print(nba_clean["gameType"].value_counts(dropna=False))


# Recalculate margins after filtering.
nba_clean["score_margin"] = (
    nba_clean["homeScore"] - nba_clean["awayScore"]
)
nba_clean["abs_margin"] = nba_clean["score_margin"].abs()

# The two tied NBA scores reported earlier are unusual.
# Print them for inspection; do not silently remove them.
nba_tied_scores = nba_clean.loc[
    nba_clean["homeScore"] == nba_clean["awayScore"],
    ["gameId", "gameDate", "hometeamName", "awayteamName",
     "homeScore", "awayScore", "attendance", "gameType"]
]

print("\nNBA games with tied recorded scores:")
print("Count:", len(nba_tied_scores))
print(nba_tied_scores.to_string(index=False))


# ------------------------------------------------------------
# PART B: MLB — check for preseason/spring-training games
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MLB: GAME-TYPE CHECK")
print("=" * 65)

print("Available MLB game types:")
print(mlb_clean["gametype"].value_counts(dropna=False))

# Search game-type labels for anything that explicitly looks like
# preseason or spring training. The previously observed labels did not
# include either, but this check verifies that against the current data.
mlb_type_text = (
    mlb_clean["gametype"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mlb_preseason_mask = mlb_type_text.str.contains(
    r"pre.?season|spring.?training|exhibition",
    regex=True,
    na=False
)

print("\nRows explicitly labeled as MLB preseason/exhibition:",
      int(mlb_preseason_mask.sum()))

if mlb_preseason_mask.any():
    print("Game types in those rows:")
    print(mlb_clean.loc[mlb_preseason_mask, "gametype"].value_counts())

    # Remove only rows explicitly identified as preseason/exhibition.
    mlb_clean = mlb_clean.loc[~mlb_preseason_mask].copy()
else:
    print("No MLB preseason/exhibition labels found; no MLB rows removed.")

# Recalculate margins in the retained MLB sample.
mlb_clean["score_margin"] = mlb_clean["hruns"] - mlb_clean["vruns"]
mlb_clean["abs_margin"] = mlb_clean["score_margin"].abs()


# ------------------------------------------------------------
# PART C: NFL — inspect weeks and possible postseason coverage
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NFL: WEEK AND POSTSEASON COVERAGE CHECK")
print("=" * 65)

print("Rows by week:")
print(nfl_clean["week"].value_counts(dropna=False).sort_index())

print("\nMinimum week:", nfl_clean["week"].min())
print("Maximum week:", nfl_clean["week"].max())

# The playoffs_home/away columns appear to be team-season indicators.
# Compare them with the week variable, but do not treat them as game-level
# playoff labels.
print("\nRows by week and playoffs_home:")
print(
    pd.crosstab(
        nfl_clean["week"],
        nfl_clean["playoffs_home"],
        dropna=False
    ).to_string()
)

print("\nRows by week and playoffs_away:")
print(
    pd.crosstab(
        nfl_clean["week"],
        nfl_clean["playoffs_away"],
        dropna=False
    ).to_string()
)

# Inspect the exact tied-score records. Ties can be legitimate NFL outcomes,
# so do not automatically delete them. Their score margin of zero is valid
# if the two scores really are equal.
nfl_tied_scores = nfl_clean.loc[
    nfl_clean["pts_win_home"] == nfl_clean["pts_loss_home"],
    ["gameDate", "week", "team_name_home", "team_name_away",
     "winner_home", "pts_win_home", "pts_loss_home",
     "weekly_attendance_home"]
]

print("\nNFL games with equal recorded scores:")
print("Count:", len(nfl_tied_scores))
print(nfl_tied_scores.to_string(index=False))

# For now, retain NFL observations as supplied. We have not yet established
# a separate game-level postseason indicator, so we cannot claim this file
# contains both regular-season and playoff games.


# ------------------------------------------------------------
# PART D: Final sample and margin summaries after preseason checks
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("UPDATED SAMPLE SUMMARY")
print("=" * 65)

for league, df, date_col, attendance_col in [
    ("NBA", nba_clean, "gameDate", "attendance"),
    ("MLB", mlb_clean, "date", "attendance"),
    ("NFL", nfl_clean, "gameDate", "weekly_attendance_home"),
]:
    print(f"\n{league}")
    print("Rows:", f"{len(df):,}")
    print("Date range:", df[date_col].min(), "to", df[date_col].max())
    print("Missing dates:", df[date_col].isna().sum())
    print("Missing attendance:", df[attendance_col].isna().sum())
    print("Nonpositive attendance:",
          (df[attendance_col] <= 0).sum())
    print("Games with missing absolute margin:", df["abs_margin"].isna().sum())
    print("10th percentile absolute margin:", df["abs_margin"].quantile(0.10))
    print("15th percentile absolute margin:", df["abs_margin"].quantile(0.15))
    print("20th percentile absolute margin:", df["abs_margin"].quantile(0.20))

print("\nCell 6 complete. No additional rows were removed beyond the explicit preseason filter.")

NBA: PRESEASON FILTER
Rows before: 66,798
Rows after: 65,216
Rows removed: 1,582

Remaining NBA game types:
gameType
Regular Season        61148
Playoffs               3973
NBA Emirates Cup         66
Play-in Tournament       28
NBA Cup                   1
Name: count, dtype: int64

NBA games with tied recorded scores:
Count: 2
  gameId            gameDate  hometeamName awayteamName  homeScore  awayScore  attendance       gameType
21000028 2010-10-29 20:30:00     Mavericks    Grizzlies         90         90     20060.0 Regular Season
20400090 2004-11-13 22:00:00 Trail Blazers      Raptors        105        105     16362.0 Regular Season

MLB: GAME-TYPE CHECK
Available MLB game types:
gametype
regular           134031
lcs                  578
divisionseries       498
worldseries          378
allstar               70
wildcard              55
playoff               14
Name: count, dtype: int64

Rows explicitly labeled as MLB preseason/exhibition: 0
No MLB preseason/exhibition labels found;

In [10]:
# CELL 7: Construct previous-game histories within each season
#
# PURPOSE
#   1. Identify each team's previous game chronologically.
#   2. Include both home and away appearances in each team's history.
#   3. Never carry a previous-game link across seasons.
#   4. Merge the previous-game information back into each game-level dataset.
#
# IMPORTANT:
#   - No close-game threshold is selected or applied.
#   - No close-game indicator is created.
#   - No regressions are run.
#   - No additional observations are intentionally deleted.
#
# OUTPUT DATAFRAMES:
#   nba_history, mlb_history, nfl_history


# ------------------------------------------------------------
# PART A: Define season identifiers
# ------------------------------------------------------------

# NBA:
# NBA seasons usually span two calendar years.
# Assign July-December games to the season beginning in that year,
# and January-June games to the season beginning in the prior year.
#
# Example:
#   October 2024 -> season_id 2024
#   February 2025 -> season_id 2024
#
# We will inspect the season assignments during validation.
nba_clean["season_id"] = np.where(
    nba_clean["gameDate"].dt.month >= 7,
    nba_clean["gameDate"].dt.year,
    nba_clean["gameDate"].dt.year - 1
)
nba_clean["season_id"] = nba_clean["season_id"].astype("Int64")


# MLB:
# Use the season field already supplied in the dataset.
mlb_clean["season_id"] = pd.to_numeric(
    mlb_clean["season"], errors="coerce"
).astype("Int64")


# NFL:
# The year field identifies the season in this dataset.
nfl_clean["season_id"] = pd.to_numeric(
    nfl_clean["year"], errors="coerce"
).astype("Int64")


# ------------------------------------------------------------
# PART B: Create a unique NFL game ID
# ------------------------------------------------------------

# Unlike the NBA and MLB datasets, the NFL dataset does not have
# a unique game identifier in the original columns.
#
# Multiple games can occur on the same date, so gameDate alone
# cannot uniquely identify a game.
#
# Create a stable row-level identifier for this cleaned dataset.
# This identifier distinguishes games but does not imply a
# chronological order for games occurring on the same date.
nfl_clean = nfl_clean.reset_index(drop=True)
nfl_clean["nfl_game_id"] = np.arange(1, len(nfl_clean) + 1)


# ------------------------------------------------------------
# PART C: Define a reusable history-construction function
# ------------------------------------------------------------

def add_previous_game_history(
    games,
    league,
    game_id_col,
    date_col,
    season_col,
    home_team_col,
    away_team_col,
    abs_margin_col="abs_margin"
):
    """
    Link each team's current game to its immediately preceding game
    within the same season.

    Each game contributes two team-game appearances:
      - the home team's appearance
      - the away team's appearance

    A team’s first appearance in each season has no eligible previous
    game. The function never links across seasons.

    The previous game can have been played at home or away.

    This function adds:
      home_prev_game_id
      home_prev_game_date
      home_prev_game_abs_margin
      home_has_prev_game_same_season

      away_prev_game_id
      away_prev_game_date
      away_prev_game_abs_margin
      away_has_prev_game_same_season

    No close-game indicator or threshold is created.
    """

    result = games.copy()

    # Validate required columns.
    required_cols = [
        game_id_col,
        date_col,
        season_col,
        home_team_col,
        away_team_col,
        abs_margin_col
    ]

    missing_cols = [
        col for col in required_cols if col not in result.columns
    ]

    if missing_cols:
        raise ValueError(
            f"{league}: required columns are missing: {missing_cols}"
        )

    # Require one identifiable row per game.
    if result[game_id_col].isna().any():
        raise ValueError(f"{league}: missing game IDs found.")

    if result[game_id_col].duplicated().any():
        raise ValueError(
            f"{league}: game IDs are not unique. Resolve this first."
        )

    if result[date_col].isna().any():
        raise ValueError(
            f"{league}: missing dates found. Resolve these before linking history."
        )

    if result[season_col].isna().any():
        raise ValueError(
            f"{league}: missing season IDs found."
        )

    if result[home_team_col].isna().any() or result[away_team_col].isna().any():
        raise ValueError(
            f"{league}: missing team identifiers found."
        )

    # --------------------------------------------------------
    # 1. Convert game rows into two team-game appearances.
    # --------------------------------------------------------

    home_appearances = pd.DataFrame({
        "game_id": result[game_id_col].to_numpy(),
        "game_date": result[date_col].to_numpy(),
        "season_id": result[season_col].to_numpy(),
        "team_id": result[home_team_col].to_numpy(),
        "venue_in_game": "home",
        "current_abs_margin": result[abs_margin_col].to_numpy()
    })

    away_appearances = pd.DataFrame({
        "game_id": result[game_id_col].to_numpy(),
        "game_date": result[date_col].to_numpy(),
        "season_id": result[season_col].to_numpy(),
        "team_id": result[away_team_col].to_numpy(),
        "venue_in_game": "away",
        "current_abs_margin": result[abs_margin_col].to_numpy()
    })

    appearances = pd.concat(
        [home_appearances, away_appearances],
        ignore_index=True
    )

    # A valid game must contain two distinct teams.
    teams_per_game = appearances.groupby("game_id")["team_id"].nunique()

    invalid_game_ids = teams_per_game[teams_per_game < 2].index

    if len(invalid_game_ids) > 0:
        raise ValueError(
            f"{league}: {len(invalid_game_ids)} games have fewer than "
            "two distinct teams."
        )

    # A team should appear only once in a given game.
    duplicate_team_game = appearances.duplicated(
        subset=["game_id", "team_id"],
        keep=False
    )

    if duplicate_team_game.any():
        print(f"\nDuplicate team-game records found in {league}:")
        print(
            appearances.loc[
                duplicate_team_game,
                ["game_id", "team_id", "game_date", "season_id"]
            ].head(15).to_string(index=False)
        )

        raise ValueError(
            f"{league}: duplicate team-game appearances must be investigated."
        )

    # --------------------------------------------------------
    # 2. Sort by team, season, date, and game ID.
    # --------------------------------------------------------

    # The game ID is a deterministic tie-breaker when games share
    # a date. It is not necessarily proof of the true order of
    # same-day games, so we will inspect those cases separately.
    appearances = appearances.sort_values(
        ["team_id", "season_id", "game_date", "game_id"],
        kind="mergesort"
    ).reset_index(drop=True)

    # Grouping by BOTH team and season prevents links across seasons.
    grouped = appearances.groupby(
        ["team_id", "season_id"],
        sort=False,
        dropna=False
    )

    # The preceding row within each team-season group is the candidate
    # previous game. The first row of each group receives missing values.
    appearances["prev_game_id"] = grouped["game_id"].shift(1)
    appearances["prev_game_date"] = grouped["game_date"].shift(1)
    appearances["prev_game_abs_margin"] = grouped[
        "current_abs_margin"
    ].shift(1)

    appearances["has_prev_game_same_season"] = (
        appearances["prev_game_id"].notna()
    )

    # --------------------------------------------------------
    # 3. Create separate lookup tables for current home/away teams.
    # --------------------------------------------------------

    home_history = appearances.loc[
        appearances["venue_in_game"] == "home",
        [
            "game_id",
            "prev_game_id",
            "prev_game_date",
            "prev_game_abs_margin",
            "has_prev_game_same_season"
        ]
    ].rename(columns={
        "game_id": game_id_col,
        "prev_game_id": "home_prev_game_id",
        "prev_game_date": "home_prev_game_date",
        "prev_game_abs_margin": "home_prev_game_abs_margin",
        "has_prev_game_same_season": "home_has_prev_game_same_season"
    })

    away_history = appearances.loc[
        appearances["venue_in_game"] == "away",
        [
            "game_id",
            "prev_game_id",
            "prev_game_date",
            "prev_game_abs_margin",
            "has_prev_game_same_season"
        ]
    ].rename(columns={
        "game_id": game_id_col,
        "prev_game_id": "away_prev_game_id",
        "prev_game_date": "away_prev_game_date",
        "prev_game_abs_margin": "away_prev_game_abs_margin",
        "has_prev_game_same_season": "away_has_prev_game_same_season"
    })

    # --------------------------------------------------------
    # 4. Merge the history columns back onto the game-level data.
    # --------------------------------------------------------

    result = result.merge(
        home_history,
        on=game_id_col,
        how="left",
        validate="one_to_one"
    )

    result = result.merge(
        away_history,
        on=game_id_col,
        how="left",
        validate="one_to_one"
    )

    # Use nullable Boolean values for the eligibility flags.
    result["home_has_prev_game_same_season"] = (
        result["home_has_prev_game_same_season"].astype("boolean")
    )

    result["away_has_prev_game_same_season"] = (
        result["away_has_prev_game_same_season"].astype("boolean")
    )

    # --------------------------------------------------------
    # 5. Basic diagnostic counts.
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print(f"{league}: PREVIOUS-GAME HISTORY DIAGNOSTICS")
    print("=" * 70)

    print("Number of games:", f"{len(result):,}")
    print("Number of team-game appearances:", f"{len(appearances):,}")

    for side in ["home", "away"]:
        flag_col = f"{side}_has_prev_game_same_season"
        margin_col = f"{side}_prev_game_abs_margin"

        print(
            f"{side.title()} team has a prior game in the same season:",
            f"{int(result[flag_col].sum()):,}"
        )
        print(
            f"{side.title()} team has no prior game in the same season:",
            f"{int((~result[flag_col]).sum()):,}"
        )

        # A linked previous game should have a recorded margin.
        # A first-of-season game should not have a previous-game margin.
        inconsistencies = (
            (result[flag_col] & result[margin_col].isna())
            | (~result[flag_col] & result[margin_col].notna())
        )

        print(
            f"{side.title()} flag/margin inconsistencies:",
            int(inconsistencies.sum())
        )

    # Count links to previous games played on the same calendar date.
    current_dates = pd.to_datetime(result[date_col]).dt.date

    for side in ["home", "away"]:
        previous_dates = pd.to_datetime(
            result[f"{side}_prev_game_date"]
        ).dt.date

        same_day = (
            previous_dates.notna()
            & (previous_dates == current_dates)
        )

        print(
            f"{side.title()} previous-game links on the same date:",
            int(same_day.sum())
        )

    return result


# ------------------------------------------------------------
# PART D: Build histories for all three leagues
# ------------------------------------------------------------

# NBA: stable team IDs identify teams across changes in display names.
nba_history = add_previous_game_history(
    games=nba_clean,
    league="NBA",
    game_id_col="gameId",
    date_col="gameDate",
    season_col="season_id",
    home_team_col="hometeamId",
    away_team_col="awayteamId",
    abs_margin_col="abs_margin"
)


# MLB: team abbreviations identify home and visiting teams.
# The season field keeps histories separate across MLB seasons.
mlb_history = add_previous_game_history(
    games=mlb_clean,
    league="MLB",
    game_id_col="gid",
    date_col="date",
    season_col="season_id",
    home_team_col="hometeam",
    away_team_col="visteam",
    abs_margin_col="abs_margin"
)


# NFL: use the unique ID created above.
# Both teams are included, so a team's prior game can be either home or away.
nfl_history = add_previous_game_history(
    games=nfl_clean,
    league="NFL",
    game_id_col="nfl_game_id",
    date_col="gameDate",
    season_col="season_id",
    home_team_col="team_name_home",
    away_team_col="team_name_away",
    abs_margin_col="abs_margin"
)

print("\nAll three team-history tables have been constructed.")
print("No close-game threshold or close-game indicator has been created.")


NBA: PREVIOUS-GAME HISTORY DIAGNOSTICS
Number of games: 65,216
Number of team-game appearances: 130,432
Home team has a prior game in the same season: 64,378
Home team has no prior game in the same season: 838
Home flag/margin inconsistencies: 0
Away team has a prior game in the same season: 64,448
Away team has no prior game in the same season: 768
Away flag/margin inconsistencies: 0
Home previous-game links on the same date: 0
Away previous-game links on the same date: 0

MLB: PREVIOUS-GAME HISTORY DIAGNOSTICS
Number of games: 135,624
Number of team-game appearances: 271,248
Home team has a prior game in the same season: 134,692
Home team has no prior game in the same season: 932
Home flag/margin inconsistencies: 0
Away team has a prior game in the same season: 134,687
Away team has no prior game in the same season: 937
Away flag/margin inconsistencies: 0
Home previous-game links on the same date: 541
Away previous-game links on the same date: 540

NFL: PREVIOUS-GAME HISTORY DIAGNOS

In [11]:
# CELL 8: Validate previous-game links and inspect examples
#
# This cell is diagnostic only.
# It does not select or apply a close-game threshold.
# It does not create a close-game indicator or run a regression.


def validate_history_links(
    df,
    league,
    game_id_col,
    date_col,
    season_col
):
    print("\n" + "=" * 70)
    print(f"{league}: PREVIOUS-GAME LINK VALIDATION")
    print("=" * 70)

    # --------------------------------------------------------
    # 1. Confirm row and identifier integrity.
    # --------------------------------------------------------

    print("Rows:", f"{len(df):,}")
    print("Unique game IDs:", f"{df[game_id_col].nunique():,}")
    print("Duplicate game IDs:", f"{df[game_id_col].duplicated().sum():,}")
    print("Missing season IDs:", f"{df[season_col].isna().sum():,}")
    print("Missing current-game dates:", f"{df[date_col].isna().sum():,}")

    # --------------------------------------------------------
    # 2. Validate each side's previous-game link.
    # --------------------------------------------------------

    current_dates = pd.to_datetime(df[date_col])

    for side in ["home", "away"]:
        previous_date_col = f"{side}_prev_game_date"
        previous_id_col = f"{side}_prev_game_id"
        previous_margin_col = f"{side}_prev_game_abs_margin"
        flag_col = f"{side}_has_prev_game_same_season"

        previous_dates = pd.to_datetime(df[previous_date_col])
        flag = df[flag_col].fillna(False)

        # A previous game cannot occur after the current game.
        previous_date_after_current = (
            previous_dates.notna()
            & (previous_dates > current_dates)
        )

        # The flag should agree with whether a previous game ID exists.
        flag_id_mismatch = (
            flag != df[previous_id_col].notna()
        )

        # If a previous game exists, its margin should be available.
        # If no previous game exists, its margin should be missing.
        flag_margin_mismatch = (
            (flag & df[previous_margin_col].isna())
            | (~flag & df[previous_margin_col].notna())
        )

        print(f"\n{side.title()} team:")
        print(
            "Previous dates later than current date:",
            int(previous_date_after_current.sum())
        )
        print(
            "Eligibility flag / previous ID mismatches:",
            int(flag_id_mismatch.sum())
        )
        print(
            "Eligibility flag / previous margin mismatches:",
            int(flag_margin_mismatch.sum())
        )

        # Check that linked previous dates do not cross seasons.
        # We use the season ID of the previous game by looking it up
        # from the original game table.
        season_lookup = (
            df[[game_id_col, season_col]]
            .drop_duplicates(subset=[game_id_col])
            .set_index(game_id_col)[season_col]
        )

        previous_seasons = df[previous_id_col].map(season_lookup)

        previous_season_mismatch = (
            flag
            & previous_seasons.notna()
            & (previous_seasons != df[season_col])
        )

        print(
            "Previous-game links crossing season IDs:",
            int(previous_season_mismatch.sum())
        )

        # Same-calendar-date links may be legitimate doubleheaders.
        # They require further review to establish actual game order.
        same_calendar_date = (
            previous_dates.notna()
            & (previous_dates.dt.date == current_dates.dt.date)
        )

        print(
            "Previous-game links on the same calendar date:",
            int(same_calendar_date.sum())
        )

    # --------------------------------------------------------
    # 3. Display first-of-season examples.
    # --------------------------------------------------------

    first_of_season = df.loc[
        ~df["home_has_prev_game_same_season"].fillna(False)
        | ~df["away_has_prev_game_same_season"].fillna(False)
    ]

    display_cols = [
        game_id_col,
        date_col,
        season_col,
        "home_has_prev_game_same_season",
        "home_prev_game_id",
        "home_prev_game_date",
        "home_prev_game_abs_margin",
        "away_has_prev_game_same_season",
        "away_prev_game_id",
        "away_prev_game_date",
        "away_prev_game_abs_margin"
    ]

    display_cols = [c for c in display_cols if c in df.columns]

    print("\nExamples where at least one team has no prior game in the season:")
    print(
        first_of_season[display_cols]
        .head(15)
        .to_string(index=False)
    )

    # --------------------------------------------------------
    # 4. Display examples where both teams have prior games.
    # --------------------------------------------------------

    both_have_previous = df.loc[
        df["home_has_prev_game_same_season"].fillna(False)
        & df["away_has_prev_game_same_season"].fillna(False)
    ]

    print("\nExamples where both teams have a prior game in the season:")
    print(
        both_have_previous[display_cols]
        .head(15)
        .to_string(index=False)
    )


# Validate all three league datasets.
validate_history_links(
    df=nba_history,
    league="NBA",
    game_id_col="gameId",
    date_col="gameDate",
    season_col="season_id"
)

validate_history_links(
    df=mlb_history,
    league="MLB",
    game_id_col="gid",
    date_col="date",
    season_col="season_id"
)

validate_history_links(
    df=nfl_history,
    league="NFL",
    game_id_col="nfl_game_id",
    date_col="gameDate",
    season_col="season_id"
)

print("\nValidation complete.")
print("No close-game threshold has been selected or applied.")


NBA: PREVIOUS-GAME LINK VALIDATION
Rows: 65,216
Unique game IDs: 65,216
Duplicate game IDs: 0
Missing season IDs: 0
Missing current-game dates: 0

Home team:
Previous dates later than current date: 0
Eligibility flag / previous ID mismatches: 0
Eligibility flag / previous margin mismatches: 0
Previous-game links crossing season IDs: 0
Previous-game links on the same calendar date: 0

Away team:
Previous dates later than current date: 0
Eligibility flag / previous ID mismatches: 0
Eligibility flag / previous margin mismatches: 0
Previous-game links crossing season IDs: 0
Previous-game links on the same calendar date: 0

Examples where at least one team has no prior game in the season:
  gameId            gameDate  season_id  home_has_prev_game_same_season  home_prev_game_id home_prev_game_date  home_prev_game_abs_margin  away_has_prev_game_same_season  away_prev_game_id away_prev_game_date  away_prev_game_abs_margin
22400075 2024-10-24 22:00:00       2024                           Fals

In [12]:

# CELL 9: Season audits and MLB same-day game investigation

# ------------------------------------------------------------
# 1. NBA season audit
# ------------------------------------------------------------

print("=" * 80)
print("NBA SEASON AUDIT")
print("=" * 80)

nba_season_audit = (
    nba_clean.groupby("season_id")
    .agg(
        first_game=("gameDate", "min"),
        last_game=("gameDate", "max"),
        games=("gameId", "count")
    )
    .sort_index()
)

print("\nFirst 10 NBA seasons:")
display(nba_season_audit.head(10))

print("\nLast 10 NBA seasons:")
display(nba_season_audit.tail(10))

nba_unusual_months = nba_clean.loc[
    nba_clean["gameDate"].dt.month.isin([7, 8, 9]),
    [
        "gameId", "gameDate", "season_id",
        "hometeamName", "awayteamName", "gameType"
    ]
].sort_values("gameDate")

print("\nNBA games in July-September:")
display(nba_unusual_months.head(30))
print("Total NBA games in July-September:", len(nba_unusual_months))


# ------------------------------------------------------------
# 2. NFL season audit and corrected season assignments
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NFL SEASON AUDIT")
print("=" * 80)

nfl_month = nfl_clean["gameDate"].dt.month

nfl_clean["season_id"] = pd.Series(
    np.where(
        nfl_month.isin([1, 2]),
        nfl_clean["gameDate"].dt.year - 1,
        nfl_clean["gameDate"].dt.year
    ),
    index=nfl_clean.index,
    dtype="Int64"
)

print("\nCalendar year versus corrected NFL season:")
display(
    pd.crosstab(
        nfl_clean["gameDate"].dt.year,
        nfl_clean["season_id"]
    )
)

nfl_season_audit = (
    nfl_clean.groupby("season_id")
    .agg(
        first_game=("gameDate", "min"),
        last_game=("gameDate", "max"),
        games=("nfl_game_id", "count")
    )
    .sort_index()
)

print("\nNFL season summary:")
display(nfl_season_audit)

nfl_unusual_months = nfl_clean.loc[
    nfl_clean["gameDate"].dt.month.isin([3, 4, 5, 6, 7, 8]),
    [
        "nfl_game_id", "gameDate", "season_id",
        "team_name_home", "team_name_away"
    ]
].sort_values("gameDate")

print("\nNFL games in March-August:")
display(nfl_unusual_months.head(30))
print("Total NFL games in March-August:", len(nfl_unusual_months))


# Rebuild NFL previous-game history with corrected season IDs.
nfl_history = add_previous_game_history(
    games=nfl_clean,
    league="NFL",
    game_id_col="nfl_game_id",
    date_col="gameDate",
    season_col="season_id",
    home_team_col="team_name_home",
    away_team_col="team_name_away",
    abs_margin_col="abs_margin"
)

print("\nNFL previous-game history rebuilt.")


# ------------------------------------------------------------
# 3. MLB same-day game / doubleheader audit
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MLB SAME-DAY GAME AUDIT")
print("=" * 80)

# MLB's date column is "date", not "gameDate".

mlb_home_appearances = (
    mlb_clean[
        [
            "gid", "date", "season_id",
            "hometeam", "visteam", "abs_margin"
        ]
    ]
    .rename(
        columns={
            "gid": "game_id",
            "date": "game_date",
            "hometeam": "team_id",
            "visteam": "opponent"
        }
    )
    .assign(venue="home")
)

mlb_away_appearances = (
    mlb_clean[
        [
            "gid", "date", "season_id",
            "visteam", "hometeam", "abs_margin"
        ]
    ]
    .rename(
        columns={
            "gid": "game_id",
            "date": "game_date",
            "visteam": "team_id",
            "hometeam": "opponent"
        }
    )
    .assign(venue="away")
)

mlb_appearances = pd.concat(
    [mlb_home_appearances, mlb_away_appearances],
    ignore_index=True
)

mlb_same_day_counts = (
    mlb_appearances
    .groupby(["team_id", "season_id", "game_date"])
    .size()
    .rename("games_on_date")
    .reset_index()
)

mlb_multiple_same_day = (
    mlb_same_day_counts[
        mlb_same_day_counts["games_on_date"] > 1
    ]
    .sort_values(["team_id", "season_id", "game_date"])
)

print(
    "Team-season-date groups with multiple games:",
    len(mlb_multiple_same_day)
)

print("\nExamples of MLB same-day games:")
display(mlb_multiple_same_day.head(30))

print("\nMLB game-ID final-character distribution:")
display(
    mlb_clean["gid"]
    .astype("string")
    .str[-1]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nCell 9 completed.")
print("No close-game threshold has been selected or applied.")



NBA SEASON AUDIT

First 10 NBA seasons:


,first_game,last_game,games
season_id,,,
1946,1947-04-14 19:00:00,1947-04-14 19:00:00,1
1948,1949-03-29 19:00:00,1949-03-29 19:00:00,1
1949,1950-02-12 19:00:00,1950-04-14 19:00:00,3
1950,1951-01-05 19:00:00,1951-01-05 19:00:00,1
1951,1951-11-29 19:00:00,1952-03-09 19:00:00,8
1952,1952-11-06 19:00:00,1952-12-30 19:00:00,5
1953,1953-12-06 19:00:00,1953-12-06 19:00:00,1
1954,1955-01-02 19:00:00,1955-03-19 19:00:00,3
1956,1956-12-07 19:00:00,1957-03-24 19:00:00,6



Last 10 NBA seasons:


,first_game,last_game,games
season_id,,,
2015,2015-10-27 20:00:00,2016-06-19 20:00:00,1316
2016,2016-10-25 22:00:00,2017-06-12 21:00:00,1308
2017,2017-10-17 20:00:00,2018-06-08 21:00:00,1312
2018,2018-10-16 20:00:00,2019-06-13 21:00:00,1312
2019,2019-10-22 22:30:00,2020-03-11 20:00:00,970
2020,2020-12-23 19:00:00,2021-06-30 21:00:00,549
2021,2021-07-01 20:30:00,2022-06-16 21:00:00,1319
2022,2022-10-18 19:30:00,2023-06-12 20:30:00,1314
2023,2023-10-24 19:30:00,2024-06-17 20:30:00,1319



NBA games in July-September:


,gameId,gameDate,season_id,hometeamName,awayteamName,gameType
5497,42000305,2021-07-01 20:30:00,2021,Bucks,Hawks,Playoffs
5496,42000306,2021-07-03 20:30:00,2021,Hawks,Bucks,Playoffs
5495,42000401,2021-07-06 21:00:00,2021,Suns,Bucks,Playoffs
5494,42000402,2021-07-08 21:00:00,2021,Suns,Bucks,Playoffs
5493,42000403,2021-07-11 20:00:00,2021,Bucks,Suns,Playoffs
5492,42000404,2021-07-14 21:00:00,2021,Bucks,Suns,Playoffs
5491,42000405,2021-07-17 21:00:00,2021,Suns,Bucks,Playoffs
5490,42000406,2021-07-20 21:00:00,2021,Bucks,Suns,Playoffs


... (8 outputs left out)

,team_id,season_id,game_date,games_on_date
166,ANA,1997,1997-07-25,2
169,ANA,1997,1997-07-28,2
357,ANA,1998,1998-08-26,2
805,ANA,2001,2001-07-22,2
1161,ANA,2003,2003-08-09,2
1824,ANA,2007,2007-08-17,2
2685,ANA,2012,2012-09-30,2
2749,ANA,2013,2013-06-08,2
3104,ANA,2015,2015-07-20,2
3159,ANA,2015,2015-09-19,2



MLB game-ID final-character distribution:


gid
0    130159
1       542
2      4923
Name: count, dtype: Int64


Cell 9 completed.
No close-game threshold has been selected or applied.


In [13]:

# CELL 10: Validate previous-game links for all three leagues

print("=" * 80)
print("NBA PREVIOUS-GAME VALIDATION")
print("=" * 80)

validate_history_links(
    df=nba_history,
    league="NBA",
    game_id_col="gameId",
    date_col="gameDate",
    season_col="season_id"
)

print("\n" + "=" * 80)
print("MLB PREVIOUS-GAME VALIDATION")
print("=" * 80)

# MLB uses "date", not "gameDate".
validate_history_links(
    df=mlb_history,
    league="MLB",
    game_id_col="gid",
    date_col="date",
    season_col="season_id"
)

print("\n" + "=" * 80)
print("NFL PREVIOUS-GAME VALIDATION")
print("=" * 80)

validate_history_links(
    df=nfl_history,
    league="NFL",
    game_id_col="nfl_game_id",
    date_col="gameDate",
    season_col="season_id"
)

print("\nCell 10 completed.")
print("No close-game threshold has been selected or applied.")



NBA PREVIOUS-GAME VALIDATION

NBA: PREVIOUS-GAME LINK VALIDATION
Rows: 65,216
Unique game IDs: 65,216
Duplicate game IDs: 0
Missing season IDs: 0
Missing current-game dates: 0

Home team:
Previous dates later than current date: 0
Eligibility flag / previous ID mismatches: 0
Eligibility flag / previous margin mismatches: 0
Previous-game links crossing season IDs: 0
Previous-game links on the same calendar date: 0

Away team:
Previous dates later than current date: 0
Eligibility flag / previous ID mismatches: 0
Eligibility flag / previous margin mismatches: 0
Previous-game links crossing season IDs: 0
Previous-game links on the same calendar date: 0

Examples where at least one team has no prior game in the season:
  gameId            gameDate  season_id  home_has_prev_game_same_season  home_prev_game_id home_prev_game_date  home_prev_game_abs_margin  away_has_prev_game_same_season  away_prev_game_id away_prev_game_date  away_prev_game_abs_margin
22400075 2024-10-24 22:00:00       2024  

In [14]:

# CELL 11: Investigate historical season and score issues
# Diagnostic only: does not drop or modify any observations.

# ------------------------------------------------------------
# 1. NBA: inspect early historical records and season fields
# ------------------------------------------------------------

print("=" * 80)
print("NBA: HISTORICAL SEASON INVESTIGATION")
print("=" * 80)

print("Available NBA date/season columns:")
print([
    col for col in nba_clean.columns
    if any(term in col.lower() for term in ["season", "year", "date"])
])

nba_early = nba_clean.loc[
    nba_clean["gameDate"].dt.year <= 1960
].sort_values("gameDate")

nba_display_cols = [
    col for col in [
        "gameId", "gameDate", "season_id", "gameType",
        "hometeamName", "awayteamName", "homeScore", "awayScore"
    ]
    if col in nba_early.columns
]

print("\nFirst 30 NBA records dated 1960 or earlier:")
display(nba_early[nba_display_cols].head(30))

print("\nNBA records by season ID before 1960:")
display(
    nba_early.groupby("season_id")
    .agg(
        first_game=("gameDate", "min"),
        last_game=("gameDate", "max"),
        games=("gameDate", "size")
    )
    .sort_index()
)


# ------------------------------------------------------------
# 2. MLB: inspect same-day games and their game IDs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MLB: DOUBLEHEADER ORDER INVESTIGATION")
print("=" * 80)

# Use examples from the same-day groups identified in Cell 9.
sample_keys = mlb_multiple_same_day.head(10)[
    ["team_id", "season_id", "game_date"]
]

sample_appearances = mlb_appearances.merge(
    sample_keys,
    on=["team_id", "season_id", "game_date"],
    how="inner"
)

print("\nSelected team appearances on dates with multiple games:")
mlb_appearance_cols = [
    col for col in [
        "team_id", "season_id", "game_date",
        "game_id", "opponent", "venue", "abs_margin"
    ]
    if col in sample_appearances.columns
]

display(
    sample_appearances[mlb_appearance_cols].sort_values(
        ["team_id", "game_date", "game_id", "venue"]
    )
)

# Inspect the original game records for those IDs.
sample_game_ids = sample_appearances["game_id"].astype(str).unique()

mlb_original_cols = [
    col for col in [
        "gid", "date", "season", "hometeam", "visteam",
        "hruns", "vruns", "attendance"
    ]
    if col in mlb_clean.columns
]

print("\nOriginal MLB records for the selected games:")
display(
    mlb_clean.loc[
        mlb_clean["gid"].astype(str).isin(sample_game_ids),
        mlb_original_cols
    ].sort_values(["date", "gid"])
)


# ------------------------------------------------------------
# 3. NFL: inspect equal-score records without dropping them
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NFL: TIES AND EQUAL-SCORE RECORD INVESTIGATION")
print("=" * 80)

# These fields are named winner/loser scores in the dataset.
# Equal scores may represent ties OR inconsistencies in the data.
# Keep all records while we investigate.

nfl_equal_score_mask = (
    nfl_clean["pts_win_home"].notna()
    & nfl_clean["pts_loss_home"].notna()
    & nfl_clean["pts_win_home"].eq(nfl_clean["pts_loss_home"])
)

nfl_equal_score_records = nfl_clean.loc[nfl_equal_score_mask].copy()

nfl_issue_cols = [
    col for col in [
        "nfl_game_id", "gameDate", "season_id",
        "team_name_home", "team_name_away",
        "pts_win_home", "pts_loss_home",
        "winner_home", "winner_away",
        "points_for_home", "points_against_home",
        "margin_of_victory_home"
    ]
    if col in nfl_equal_score_records.columns
]

print("Number of records with equal winning/losing score fields:",
      len(nfl_equal_score_records))

print("\nEqual-score records:")
display(
    nfl_equal_score_records[nfl_issue_cols].sort_values("gameDate")
)

print("\nNumber of rows in nfl_clean:", len(nfl_clean))
print("No NFL rows have been dropped or modified by this cell.")
print("NFL ties and possible ties remain included.")
print("No close-game threshold has been selected or applied.")


NBA: HISTORICAL SEASON INVESTIGATION
Available NBA date/season columns:
['gameDate', 'season_id']



First 30 NBA records dated 1960 or earlier:


,gameId,gameDate,season_id,gameType,hometeamName,awayteamName,homeScore,awayScore
71807,44600212,1947-04-14 19:00:00,1946,Playoffs,Knicks,Warriors,53,72
71700,44800312,1949-03-29 19:00:00,1948,Playoffs,Lakers,Royals,67,55
71610,24900418,1950-02-12 19:00:00,1949,Regular Season,Lakers,Celtics,88,70
71568,44900411,1950-04-08 19:00:00,1949,Playoffs,Nationals,Lakers,66,68
71566,44900413,1950-04-14 19:00:00,1949,Playoffs,Lakers,Nationals,91,77
71477,25000173,1951-01-05 19:00:00,1950,Regular Season,Lakers,Celtics,69,54
71302,25100068,1951-11-29 19:00:00,1951,Regular Season,Nationals,Pistons,82,78
71298,25100080,1951-12-02 19:00:00,1951,Regular Season,Pistons,Lakers,63,61
71293,25100084,1951-12-05 19:00:00,1951,Regular Season,Celtics,Pistons,88,91
71221,25100199,1952-01-24 19:00:00,1951,Regular Season,Pistons,Royals,94,90



NBA records by season ID before 1960:


,first_game,last_game,games
season_id,,,
1946,1947-04-14 19:00:00,1947-04-14 19:00:00,1
1948,1949-03-29 19:00:00,1949-03-29 19:00:00,1
1949,1950-02-12 19:00:00,1950-04-14 19:00:00,3
1950,1951-01-05 19:00:00,1951-01-05 19:00:00,1
1951,1951-11-29 19:00:00,1952-03-09 19:00:00,8
1952,1952-11-06 19:00:00,1952-12-30 19:00:00,5
1953,1953-12-06 19:00:00,1953-12-06 19:00:00,1
1954,1955-01-02 19:00:00,1955-03-19 19:00:00,3
1956,1956-12-07 19:00:00,1957-03-24 19:00:00,6



MLB: DOUBLEHEADER ORDER INVESTIGATION

Selected team appearances on dates with multiple games:


... (2 outputs left out)

,gid,date,season,hometeam,visteam,hruns,vruns,attendance
157798,BOS199707251,1997-07-25,1997,BOS,ANA,4,5,20427.0
157799,BOS199707252,1997-07-25,1997,BOS,ANA,5,8,24631.0
157844,CLE199707281,1997-07-28,1997,CLE,ANA,0,2,43033.0
157845,CLE199707282,1997-07-28,1997,CLE,ANA,7,10,42857.0
160682,NYA199808261,1998-08-26,1998,NYA,ANA,4,6,20077.0
160683,NYA199808262,1998-08-26,1998,NYA,ANA,7,6,28837.0
167538,BAL200107221,2001-07-22,2001,BAL,ANA,4,9,40543.0
167539,BAL200107222,2001-07-22,2001,BAL,ANA,5,1,38423.0
172736,CLE200308091,2003-08-09,2003,CLE,ANA,3,2,26391.0
172737,CLE200308092,2003-08-09,2003,CLE,ANA,3,2,20116.0



NFL: TIES AND EQUAL-SCORE RECORD INVESTIGATION
Number of records with equal winning/losing score fields: 10

Equal-score records:


,nfl_game_id,gameDate,season_id,team_name_home,team_name_away,pts_win_home,pts_loss_home,winner_home,winner_away,points_for_home,points_against_home,margin_of_victory_home
431,432,2002-11-10,2002,Pittsburgh Steelers,Atlanta Falcons,34,34,Pittsburgh Steelers,Pittsburgh Steelers,352,212,8.8
1820,1821,2008-11-16,2008,Cincinnati Bengals,Philadelphia Eagles,13,13,Philadelphia Eagles,Philadelphia Eagles,380,385,-0.3
3004,3005,2012-11-11,2012,San Francisco 49ers,St. Louis Rams,24,24,San Francisco 49ers,San Francisco 49ers,380,229,9.4
3141,3142,2013-11-24,2013,Green Bay Packers,Minnesota Vikings,26,26,Minnesota Vikings,Minnesota Vikings,433,336,6.1
3354,3355,2014-10-12,2014,Cincinnati Bengals,Carolina Panthers,37,37,Cincinnati Bengals,Cincinnati Bengals,430,305,7.8
3819,3820,2016-10-23,2016,Arizona Cardinals,Seattle Seahawks,6,6,Seattle Seahawks,Seattle Seahawks,489,313,11.0
3866,3867,2016-10-30,2016,Cincinnati Bengals,Washington Redskins,27,27,Washington Redskins,Washington Redskins,419,279,8.8
4352,4353,2018-09-09,2018,Cleveland Browns,Pittsburgh Steelers,21,21,Pittsburgh Steelers,Pittsburgh Steelers,234,410,-11.0
4385,4386,2018-09-16,2018,Green Bay Packers,Minnesota Vikings,29,29,Minnesota Vikings,Minnesota Vikings,320,384,-4.0
4552,4553,2019-09-08,2019,Arizona Cardinals,Detroit Lions,27,27,Detroit Lions,Detroit Lions,225,425,-12.5



Number of rows in nfl_clean: 4808
No NFL rows have been dropped or modified by this cell.
NFL ties and possible ties remain included.
No close-game threshold has been selected or applied.


In [15]:
# CELL 12: Validate NBA seasons, MLB doubleheader ordering, and NFL score fields
# Diagnostic only: does not drop observations or alter the working datasets.

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. NBA: verify historical season assignments
# ------------------------------------------------------------

print("=" * 80)
print("NBA: SEASON ASSIGNMENT CHECK")
print("=" * 80)

# The NBA season label generally refers to the year the season began.
# For example, the 1946-47 season is labeled 1946.
# Check whether the existing July-boundary rule agrees with that convention.

nba_season_check = nba_clean.copy()
nba_season_check["date_year"] = nba_season_check["gameDate"].dt.year
nba_season_check["date_month"] = nba_season_check["gameDate"].dt.month

nba_season_check["expected_season_july_rule"] = np.where(
    nba_season_check["date_month"] >= 7,
    nba_season_check["date_year"],
    nba_season_check["date_year"] - 1
)

nba_season_check["season_rule_mismatch"] = (
    nba_season_check["season_id"]
    != nba_season_check["expected_season_july_rule"]
)

print("Rows:", len(nba_season_check))
print(
    "Rows that differ from the July-boundary rule:",
    int(nba_season_check["season_rule_mismatch"].sum())
)

print("\nHistorical NBA games with season labels:")
display(
    nba_season_check.loc[
        nba_season_check["date_year"] <= 1960,
        [
            "gameId", "gameDate", "season_id",
            "gameType", "hometeamName", "awayteamName",
            "season_rule_mismatch"
        ]
    ].sort_values("gameDate").head(40)
)

print("\nNBA season date ranges for seasons with fewer than 20 recorded games:")
nba_season_summary = (
    nba_clean.groupby("season_id")
    .agg(
        first_game=("gameDate", "min"),
        last_game=("gameDate", "max"),
        games=("gameDate", "size")
    )
)

display(
    nba_season_summary.loc[nba_season_summary["games"] < 20]
    .sort_index()
)


# ------------------------------------------------------------
# 2. MLB: validate game-number suffixes within doubleheaders
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MLB: DOUBLEHEADER GAME ORDER CHECK")
print("=" * 80)

# MLB game IDs commonly end in a game number:
# 1 = first game, 2 = second game.
# Validate whether the suffix is consistent within same-day team appearances.

mlb_appearances_check = mlb_appearances.copy()
mlb_appearances_check["game_id"] = (
    mlb_appearances_check["game_id"].astype(str)
)

mlb_appearances_check["game_number_suffix"] = (
    mlb_appearances_check["game_id"].str[-1]
)

mlb_appearances_check["valid_numeric_suffix"] = (
    mlb_appearances_check["game_number_suffix"].isin(["0", "1", "2"])
)

mlb_same_day_counts = (
    mlb_appearances_check.groupby(
        ["team_id", "season_id", "game_date"]
    )
    .agg(
        games_that_day=("game_id", "nunique"),
        distinct_suffixes=("game_number_suffix", "nunique"),
        suffixes=("game_number_suffix", lambda x: ", ".join(sorted(x)))
    )
    .reset_index()
)

mlb_doubleheader_groups = mlb_same_day_counts.loc[
    mlb_same_day_counts["games_that_day"] > 1
].copy()

print("Team-season-date groups with multiple games:",
      len(mlb_doubleheader_groups))

print("\nDistribution of games per team on a single date:")
display(
    mlb_doubleheader_groups["games_that_day"]
    .value_counts()
    .sort_index()
    .rename_axis("games_that_day")
    .to_frame("number_of_groups")
)

print("\nExamples of multiple-game groups with suffix information:")
display(mlb_doubleheader_groups.head(30))

# Check whether the suffixes 1 and 2 distinguish games within each group.
mlb_suffix_check = (
    mlb_appearances_check.groupby(
        ["team_id", "season_id", "game_date"]
    )["game_number_suffix"]
    .agg(lambda x: set(x))
    .reset_index(name="suffix_set")
)

mlb_suffix_check = mlb_suffix_check.merge(
    mlb_same_day_counts[
        ["team_id", "season_id", "game_date", "games_that_day"]
    ],
    on=["team_id", "season_id", "game_date"],
    how="left"
)

mlb_suffix_check = mlb_suffix_check.loc[
    mlb_suffix_check["games_that_day"] > 1
].copy()

print("\nMultiple-game groups that do not have distinct suffixes for each game:")
display(
    mlb_suffix_check.loc[
        mlb_suffix_check.apply(
            lambda row: len(row["suffix_set"]) != row["games_that_day"],
            axis=1
        )
    ].head(30)
)


# ------------------------------------------------------------
# 3. NFL: inspect all available source columns and equal-score rows
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NFL: SOURCE COLUMN INVESTIGATION")
print("=" * 80)

print("All columns in nfl_clean:")
print(nfl_clean.columns.tolist())

print("\nColumns that may contain game-level scores or results:")
nfl_score_related_cols = [
    col for col in nfl_clean.columns
    if any(
        term in col.lower()
        for term in [
            "score", "pts", "point", "winner", "loser",
            "win", "loss", "margin", "team_name"
        ]
    )
]
print(nfl_score_related_cols)

# Keep all records. This only displays rows with equal winner/loser score fields.
nfl_equal_score_mask = (
    nfl_clean["pts_win_home"].notna()
    & nfl_clean["pts_loss_home"].notna()
    & nfl_clean["pts_win_home"].eq(nfl_clean["pts_loss_home"])
)

nfl_equal_score_records = nfl_clean.loc[nfl_equal_score_mask].copy()

print("\nEqual-score records, including all score/result-related columns:")
display(
    nfl_equal_score_records[
        [
            col for col in [
                "nfl_game_id", "gameDate", "season_id",
                "team_name_home", "team_name_away",
                "pts_win_home", "pts_loss_home",
                "pts_win_away", "pts_loss_away",
                "winner_home", "winner_away",
                "points_for_home", "points_against_home",
                "margin_of_victory_home",
                "total_home", "total_away"
            ]
            if col in nfl_equal_score_records.columns
        ]
    ].sort_values("gameDate")
)

print("\nTotal NFL rows:", len(nfl_clean))
print("Equal-score rows retained:", len(nfl_equal_score_records))
print("No NFL observations have been dropped or modified.")
print("No close-game threshold has been selected or applied.")

NBA: SEASON ASSIGNMENT CHECK
Rows: 65216
Rows that differ from the July-boundary rule: 0

Historical NBA games with season labels:


,gameId,gameDate,season_id,gameType,hometeamName,awayteamName,season_rule_mismatch
71807,44600212,1947-04-14 19:00:00,1946,Playoffs,Knicks,Warriors,False
71700,44800312,1949-03-29 19:00:00,1948,Playoffs,Lakers,Royals,False
71610,24900418,1950-02-12 19:00:00,1949,Regular Season,Lakers,Celtics,False
71568,44900411,1950-04-08 19:00:00,1949,Playoffs,Nationals,Lakers,False
71566,44900413,1950-04-14 19:00:00,1949,Playoffs,Lakers,Nationals,False
71477,25000173,1951-01-05 19:00:00,1950,Regular Season,Lakers,Celtics,False
71302,25100068,1951-11-29 19:00:00,1951,Regular Season,Nationals,Pistons,False
71298,25100080,1951-12-02 19:00:00,1951,Regular Season,Pistons,Lakers,False
71293,25100084,1951-12-05 19:00:00,1951,Regular Season,Celtics,Pistons,False
71221,25100199,1952-01-24 19:00:00,1951,Regular Season,Pistons,Royals,False



NBA season date ranges for seasons with fewer than 20 recorded games:


,first_game,last_game,games
season_id,,,
1946,1947-04-14 19:00:00,1947-04-14 19:00:00,1
1948,1949-03-29 19:00:00,1949-03-29 19:00:00,1
1949,1950-02-12 19:00:00,1950-04-14 19:00:00,3
1950,1951-01-05 19:00:00,1951-01-05 19:00:00,1
1951,1951-11-29 19:00:00,1952-03-09 19:00:00,8
1952,1952-11-06 19:00:00,1952-12-30 19:00:00,5
1953,1953-12-06 19:00:00,1953-12-06 19:00:00,1
1954,1955-01-02 19:00:00,1955-03-19 19:00:00,3
1956,1956-12-07 19:00:00,1957-03-24 19:00:00,6



MLB: DOUBLEHEADER GAME ORDER CHECK
Team-season-date groups with multiple games: 1081

Distribution of games per team on a single date:


,number_of_groups
games_that_day,
2,1081


... (3 outputs left out)

,team_id,season_id,game_date,suffix_set,games_that_day



NFL: SOURCE COLUMN INVESTIGATION
All columns in nfl_clean:
['city', 'team_name_home', 'year', 'total_home', 'home_home', 'away_home', 'week', 'weekly_attendance_home', 'wins_home', 'loss_home', 'points_for_home', 'points_against_home', 'points_differential_home', 'margin_of_victory_home', 'strength_of_schedule_home', 'simple_rating_home', 'offensive_ranking_home', 'defensive_ranking_home', 'playoffs_home', 'sb_winner_home', 'winner_home', 'tie_home', 'day_home', 'date_home', 'time_home', 'pts_win_home', 'pts_loss_home', 'yds_win_home', 'turnovers_win_home', 'yds_loss_home', 'turnovers_loss_home', 'home_team_name_home', 'home_team_city_home', 'away_team_name_home', 'away_team_city_home', 'team_name_away', 'total_away', 'home_away', 'away_away', 'wins_away', 'loss_away', 'points_for_away', 'points_against_away', 'points_differential_away', 'margin_of_victory_away', 'strength_of_schedule_away', 'simple_rating_away', 'offensive_ranking_away', 'defensive_ranking_away', 'playoffs_away', 'sb

,nfl_game_id,gameDate,season_id,team_name_home,team_name_away,pts_win_home,pts_loss_home,pts_win_away,pts_loss_away,winner_home,winner_away,points_for_home,points_against_home,margin_of_victory_home,total_home,total_away
431,432,2002-11-10,2002,Pittsburgh Steelers,Atlanta Falcons,34,34,34,34,Pittsburgh Steelers,Pittsburgh Steelers,352,212,8.8,1055127,917882
1820,1821,2008-11-16,2008,Cincinnati Bengals,Philadelphia Eagles,13,13,13,13,Philadelphia Eagles,Philadelphia Eagles,380,385,-0.3,1083308,1127787
3004,3005,2012-11-11,2012,San Francisco 49ers,St. Louis Rams,24,24,24,24,San Francisco 49ers,San Francisco 49ers,380,229,9.4,1065296,1002051
3141,3142,2013-11-24,2013,Green Bay Packers,Minnesota Vikings,26,26,26,26,Minnesota Vikings,Minnesota Vikings,433,336,6.1,1106110,1019702
3354,3355,2014-10-12,2014,Cincinnati Bengals,Carolina Panthers,37,37,37,37,Cincinnati Bengals,Cincinnati Bengals,430,305,7.8,1001806,1112229
3819,3820,2016-10-23,2016,Arizona Cardinals,Seattle Seahawks,6,6,6,6,Seattle Seahawks,Seattle Seahawks,489,313,11.0,1028446,1097931
3866,3867,2016-10-30,2016,Cincinnati Bengals,Washington Redskins,27,27,27,27,Washington Redskins,Washington Redskins,419,279,8.8,1028118,1196087
4352,4353,2018-09-09,2018,Cleveland Browns,Pittsburgh Steelers,21,21,21,21,Pittsburgh Steelers,Pittsburgh Steelers,234,410,-11.0,968007,1035491
4385,4386,2018-09-16,2018,Green Bay Packers,Minnesota Vikings,29,29,29,29,Minnesota Vikings,Minnesota Vikings,320,384,-4.0,1183556,1099905
4552,4553,2019-09-08,2019,Arizona Cardinals,Detroit Lions,27,27,27,27,Detroit Lions,Detroit Lions,225,425,-12.5,1018127,1054328



Total NFL rows: 4808
Equal-score rows retained: 10
No NFL observations have been dropped or modified.
No close-game threshold has been selected or applied.
